# Multilayer Perceptron

Binary classification on California housing: predict whether median house value is above the median.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_curve, auc)

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

threshold = data_df['median_house_value'].median()
y = (data_df['median_house_value'] > threshold).astype(int).to_numpy()
X = data_df.drop(columns=['median_house_value']).to_numpy()

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, shuffle=True, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(np.bincount(y_train), np.bincount(y_test))

## Baseline MLP

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                    solver='adam', max_iter=200, random_state=42)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
y_proba = mlp.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Below', 'Above']))

plt.figure(figsize=(8, 4))
plt.plot(mlp.loss_curve_)
plt.xlabel('iteration')
plt.ylabel('loss')
plt.grid(alpha=0.3)
plt.show()

## Architecture comparison

In [ ]:
archs = [(16,), (64,), (128,), (32, 16), (64, 32), (128, 64, 32)]
results = []

for arch in archs:
    m = MLPClassifier(hidden_layer_sizes=arch, activation='relu',
                      solver='adam', max_iter=200, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    results.append((str(arch), tr, te, m.loss_curve_))
    print(f"{str(arch):<20} train={tr:.4f} test={te:.4f}")

plt.figure(figsize=(9, 5))
for label, _, _, curve in results:
    plt.plot(curve, label=label)
plt.xlabel('iteration')
plt.ylabel('loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Activation function comparison

In [ ]:
activations = ['identity', 'logistic', 'tanh', 'relu']
act_results = []

for act in activations:
    m = MLPClassifier(hidden_layer_sizes=(64, 32), activation=act,
                      solver='adam', max_iter=200, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test, m.predict(X_test))
    act_results.append((act, tr, te, m.loss_curve_))
    print(f"{act:<10} train={tr:.4f} test={te:.4f}")

plt.figure(figsize=(9, 5))
for label, _, _, curve in act_results:
    plt.plot(curve, label=label)
plt.xlabel('iteration')
plt.ylabel('loss')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## From-scratch MLP (one hidden layer)

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

class MLPScratch:
    def __init__(self, hidden=32, lr=0.01, epochs=200, batch=128, seed=42):
        self.hidden = hidden
        self.lr = lr
        self.epochs = epochs
        self.batch = batch
        self.seed = seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        h = self.hidden
        self.W1 = rng.normal(0, np.sqrt(2.0 / d), size=(d, h))
        self.b1 = np.zeros(h)
        self.W2 = rng.normal(0, np.sqrt(2.0 / h), size=(h, 1))
        self.b2 = np.zeros(1)
        y = y.reshape(-1, 1).astype(float)
        self.loss_ = []
        for _ in range(self.epochs):
            idx = rng.permutation(n)
            ep_loss = 0.0
            for s in range(0, n, self.batch):
                b_idx = idx[s:s + self.batch]
                xb, yb = X[b_idx], y[b_idx]
                z1 = xb @ self.W1 + self.b1
                a1 = relu(z1)
                z2 = a1 @ self.W2 + self.b2
                p = sigmoid(z2)
                loss = -np.mean(yb * np.log(p + 1e-12) + (1 - yb) * np.log(1 - p + 1e-12))
                ep_loss += loss * len(b_idx)
                dz2 = (p - yb) / len(b_idx)
                dW2 = a1.T @ dz2
                db2 = dz2.sum(axis=0)
                da1 = dz2 @ self.W2.T
                dz1 = da1 * relu_grad(z1)
                dW1 = xb.T @ dz1
                db1 = dz1.sum(axis=0)
                self.W2 -= self.lr * dW2
                self.b2 -= self.lr * db2
                self.W1 -= self.lr * dW1
                self.b1 -= self.lr * db1
            self.loss_.append(ep_loss / n)
        return self

    def predict_proba(self, X):
        a1 = relu(X @ self.W1 + self.b1)
        return sigmoid(a1 @ self.W2 + self.b2).ravel()

    def predict(self, X, t=0.5):
        return (self.predict_proba(X) >= t).astype(int)

scratch = MLPScratch(hidden=64, lr=0.05, epochs=100, batch=128).fit(X_train, y_train)
print(f"Scratch train acc: {accuracy_score(y_train, scratch.predict(X_train)):.4f}")
print(f"Scratch test acc:  {accuracy_score(y_test, scratch.predict(X_test)):.4f}")

plt.figure(figsize=(8, 4))
plt.plot(scratch.loss_)
plt.xlabel('epoch')
plt.ylabel('log loss')
plt.grid(alpha=0.3)
plt.show()

## ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.grid(alpha=0.3)
plt.show()